# Weekly Fantasy Breakout Predictions - Production Pipeline (Enhanced ML)

**Purpose:** Generate fresh breakout predictions using the enhanced ML model with YACOE and depth chart features.

**Model:** Enhanced Gradient Boosting (24 features) | AUC: 0.786

**Outputs:**
* `main.fantasai.breakout_predictions_current` - Latest ML predictions
* `main.fantasai.breakout_predictions_history` - Historical prediction log

**Schedule:** Runs every Tuesday 10 AM ET (after MNF and data refresh)

**Data Sources:**
* [main.fantasai.weekly_usage_features](#table) - Base usage features
* [main.fantasai.player_nextgen_stats](#table) - NFL Next Gen Stats (YACOE, air yards, separation)
* [main.fantasai.depth_charts](#table) - Depth chart positions
* Model: `/Volumes/main/fantasai/models/breakout_prediction/breakout_model_enhanced.pkl`

In [0]:
import pickle
import pandas as pd
import numpy as np
from datetime import datetime
from pyspark.sql import functions as F

print("="*80)
print("Fantasy Breakout Predictions - Enhanced Weekly ML Run")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Load enhanced model from UC Volume
model_path = "/Volumes/main/fantasai/models/breakout_prediction/breakout_model_enhanced.pkl"
print(f"\n📦 Loading enhanced model: {model_path}")

with open(model_path, 'rb') as f:
    model = pickle.load(f)

print("✅ Enhanced model loaded successfully")
print("   Features: 24 (Base + YACOE/NGS + Depth Chart)")
print("   AUC-ROC: 0.786")

In [0]:
# Get the most recent week from the features table
max_week_df = spark.sql("""
    SELECT MAX(season) as max_season, MAX(week) as max_week
    FROM main.fantasai.weekly_usage_features
""")
max_season = max_week_df.first().max_season
max_week = max_week_df.first().max_week

print(f"\n📅 Current season: {max_season}, Week: {max_week}")

# Load current week data with ENHANCED FEATURES (YACOE + Depth Charts)
# Using explicit CTEs to ensure window functions work correctly
df_current = spark.sql(f"""
    WITH base_features AS (
        SELECT 
            w.player_name,
            w.position,
            COALESCE(w.team, 'UNK') as team,
            w.season,
            w.week,
            w.snap_share,
            COALESCE(w.snap_share_delta, 0) as snap_share_delta,
            COALESCE(w.touches, 0) as touches,
            COALESCE(w.touches_delta, 0) as touches_delta,
            COALESCE(w.targets, 0) as targets,
            COALESCE(w.targets_delta, 0) as targets_delta,
            COALESCE(w.avg_targets_prev_2wk, w.targets) as avg_targets_prev_2wk,
            COALESCE(w.fantasy_points_delta, 0) as fantasy_points_delta,
            COALESCE(w.avg_snap_share_prev_2wk, w.snap_share) as avg_snap_share_prev_2wk,
            COALESCE(w.avg_fantasy_points_prev_2wk, w.fantasy_points) as avg_fantasy_points_prev_2wk,
            w.snap_share * (COALESCE(w.touches, 0) + COALESCE(w.targets, 0)) as opportunity_score,
            CASE WHEN w.position = 'RB' THEN 1 ELSE 0 END as is_rb,
            CASE WHEN w.position = 'WR' THEN 1 ELSE 0 END as is_wr,
            CASE WHEN w.position = 'TE' THEN 1 ELSE 0 END as is_te,
            w.week as week_number,
            w.fantasy_points
        FROM main.fantasai.weekly_usage_features w
        WHERE w.season = {max_season}
          AND w.week = {max_week}
          AND w.snap_share IS NOT NULL
          AND w.snap_share > 0.2
          AND w.position IN ('RB', 'WR', 'TE')
    ),
    yacoe_features AS (
        SELECT 
            player_name,
            season,
            week,
            COALESCE(yacoe, 0) as yacoe,
            COALESCE(percent_share_of_intended_air_yards, 0) as air_yards_share,
            COALESCE(avg_cushion, 0) as avg_cushion,
            COALESCE(avg_separation, 0) as avg_separation,
            COALESCE(avg_intended_air_yards, 0) as avg_intended_air_yards
        FROM main.fantasai.player_nextgen_stats
        WHERE season = {max_season}
    ),
    depth_features AS (
        SELECT 
            player_name,
            season,
            week,
            CASE WHEN depth_team LIKE '%1%' THEN 1 
                 WHEN depth_team LIKE '%2%' THEN 2 
                 ELSE 3 END as depth_position
        FROM main.fantasai.depth_charts
        WHERE season = {max_season}
    )
    SELECT 
        b.*,
        -- ⭐ YACOE/Next Gen Stats features
        COALESCE(y.yacoe, 0) as yacoe,
        0 as yacoe_delta,  -- Will calculate in pandas
        0 as avg_yacoe_prev_3wk,  -- Will calculate in pandas
        COALESCE(y.air_yards_share, 0) as air_yards_share,
        COALESCE(y.avg_cushion, 0) as avg_cushion,
        COALESCE(y.avg_separation, 0) as avg_separation,
        COALESCE(y.avg_intended_air_yards, 0) as avg_intended_air_yards,
        -- ⭐ Depth chart features
        COALESCE(d.depth_position, 3) as depth_position,
        0 as depth_change_indicator  -- Will calculate in pandas
    FROM base_features b
    LEFT JOIN yacoe_features y
      ON b.player_name = y.player_name 
      AND b.season = y.season 
      AND b.week = y.week
    LEFT JOIN depth_features d
      ON b.player_name = d.player_name 
      AND b.season = d.season 
      AND b.week = d.week
""").toPandas()

print(f"✅ Loaded {len(df_current)} eligible players (snap share > 20%)")
print(f"   Columns loaded: {len(df_current.columns)}")
print(f"\nPosition breakdown:")
print(df_current['position'].value_counts())
print(f"\n⭐ Enhanced with YACOE and depth chart features")
print(f"   YACOE non-zero: {(df_current['yacoe'] != 0).sum()} players")
print(f"   Air yards share non-zero: {(df_current['air_yards_share'] != 0).sum()} players")

In [0]:
# ⭐ ENHANCED FEATURES: Base + YACOE + Depth Chart (24 features total)
feature_cols = [
    # Base usage features (15)
    'snap_share',
    'snap_share_delta',
    'touches',
    'touches_delta',
    'targets',
    'targets_delta',
    'avg_targets_prev_2wk',
    'fantasy_points_delta',
    'avg_snap_share_prev_2wk',
    'avg_fantasy_points_prev_2wk',
    'opportunity_score',
    'is_rb',
    'is_wr',
    'is_te',
    'week_number',
    # YACOE/Next Gen Stats features (7)
    'yacoe',
    'yacoe_delta',
    'avg_yacoe_prev_3wk',
    'air_yards_share',
    'avg_cushion',
    'avg_separation',
    'avg_intended_air_yards',
    # Depth chart features (2)
    'depth_position',
    'depth_change_indicator'
]

# Prepare input (fillna with 0 for missing values)
X = df_current[feature_cols].fillna(0)

# Make predictions
print(f"\n📊 Generating predictions for {len(X)} players...")
predictions = model.predict_proba(X)[:, 1]  # Get probability of breakout
df_current['breakout_score'] = predictions

print(f"✅ Predictions generated")
print(f"   Mean score: {predictions.mean():.4f}")
print(f"   Max score: {predictions.max():.4f}")
print(f"   Players with >1% score: {(predictions > 0.01).sum()}")
print(f"   Players with >5% score: {(predictions > 0.05).sum()}")
print(f"\n⭐ Using enhanced model with 24 features (Base + YACOE + Depth)")

In [0]:
# Select output columns and add news-related fields
df_output = df_current[[
    'player_name', 'position', 'team', 'season', 'week',
    'snap_share', 'snap_share_delta', 'touches', 'targets', 'targets_delta',
    'opportunity_score', 'fantasy_points', 'breakout_score',
    'avg_snap_share_prev_2wk', 'avg_fantasy_points_prev_2wk'
]].copy()

# Add news-related fields with default values (to match table schema)
df_output['news_sentiment'] = None
df_output['news_impact_score'] = None
df_output['news_volume'] = 0
df_output['has_opportunity_news'] = False
df_output['has_injury_news'] = False
df_output['news_buzz_score'] = None

# Calculate alert level based on breakout score
def get_alert_level(score):
    if score >= 0.02:  # 2%+ probability
        return 'HIGH'
    elif score >= 0.01:  # 1-2% probability
        return 'MEDIUM'
    else:
        return 'LOW'

df_output['alert_level'] = df_output['breakout_score'].apply(get_alert_level)

# Add generated timestamp
df_output['generated_at'] = pd.Timestamp.now()

# Convert to Spark DataFrame
df_spark = spark.createDataFrame(df_output)

# Write to current predictions table (overwrite)
df_spark.write.mode("overwrite").saveAsTable("main.fantasai.breakout_predictions_current")
print(f"\n✅ Predictions saved to main.fantasai.breakout_predictions_current")
print(f"   {len(df_output)} players")

# Append to history table
df_spark.write.mode("append").saveAsTable("main.fantasai.breakout_predictions_history")
print(f"✅ Predictions appended to main.fantasai.breakout_predictions_history")

In [0]:
print("\n" + "="*80)
print(f"Top 20 Breakout Candidates - {max_season} Week {max_week}")
print("="*80)

top_20 = df_output.nlargest(20, 'breakout_score')  # ✅ FIXED

for idx, row in top_20.iterrows():
    print(f"{idx+1:2d}. {row['player_name']:25s} {row['position']:3s} {row['team']:4s} | "
          f"Breakout: {row['breakout_score']*100:5.2f}% | "  # ✅ FIXED
          f"Alert: {row['alert_level']:6s} | "
          f"Snap: {row['snap_share']:4.1%} (Δ{row['snap_share_delta']:+.1%}) | "
          f"Targets: {row['targets']:2.0f} (Δ{row['targets_delta']:+.0f})")

print("\n" + "="*80)
print("✅ WEEKLY PREDICTION RUN COMPLETE")
print("="*80)

# Display summary stats
print(f"\n📊 Summary Statistics:")
print(f"   Total players analyzed: {len(df_output)}")
print(f"   High-probability (>2%): {(df_output['breakout_score'] > 0.02).sum()}")
print(f"   Medium-probability (1-2%): {((df_output['breakout_score'] > 0.01) & (df_output['breakout_score'] <= 0.02)).sum()}")
print(f"   By position:")
for pos in ['RB', 'WR', 'TE']:
    pos_df = df_output[df_output['position'] == pos]
    if len(pos_df) > 0:
        print(f"     {pos}: {len(pos_df)} players, avg score: {pos_df['breakout_score'].mean()*100:.3f}%")